In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import MiniBatchKMeans

SEED = 1660

## Читаем данные

In [ ]:
df = pd.read_parquet("datasets/final_train.parquet")

In [ ]:
X = df.drop(columns=["GameID", "Elo"])
Y = df["Elo"]

## Делаем PCA

In [ ]:
X_scaled = (X - X.mean()) / X.std()

In [ ]:
pca = PCA(n_components=2, random_state=SEED)
df_clust = pd.DataFrame(pca.fit_transform(X_scaled))
print(pca.explained_variance_ratio_ * 100)

## График

In [ ]:
df_clust["Y"] = Y.values

In [ ]:
cluster_model = MiniBatchKMeans(n_clusters=300, batch_size=10_000, random_state=SEED)
df_clust["cluster"] = cluster_model.fit_predict(df_clust[[0, 1]])

In [ ]:
df_clust["color"] = df_clust["cluster"].map(
    df_clust
    .groupby("cluster")
    .agg({"Y": "mean"})
    .squeeze().to_dict()
)

In [ ]:
fig = px.scatter(
    df_clust,
    x=0,
    y=1,
    color="color",
    color_continuous_scale=["darkred", "red", "orange", "yellow", "lime", "green"]
)

fig.data[0].marker.size=3

fig.update_layout(
    template="plotly_dark",
    height=900,
    width=900
)

fig.show()

## Метрики

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

In [ ]:
r2_score(df_clust["Y"], df_clust["color"]) * 100

In [ ]:
mean_absolute_error(df_clust["Y"], df_clust["color"])

## TSNE

![alt text](presentation/images/tsne.png "TSNE Results")